**Finding real and complex roots numerically.**
This notebook compares two root-finding strategies:

- **`scipy.optimize.fsolve`** is a local Newton-based solver. Given a
  starting point it converges to the nearest root, but finds only one
  root per call. It operates entirely in the reals.
- **`numpy.polynomial.Polynomial.roots`** finds all roots of a polynomial
  simultaneously, including complex ones, by computing eigenvalues of the
  companion matrix.

The key insight is that a non-polynomial function $f(x)$ can sometimes
be transformed into a polynomial by algebraic manipulation (rationalizing,
substitution). The polynomial may have more roots than the original
function - extraneous roots introduced by the manipulation - so the
results must always be verified by substituting back into $f$.

In [ ]:
"""complex_roots.ipynb"""

# Cell 01 - f(x) = x^4 + x - 1
# A degree-4 polynomial has exactly 4 roots (real or complex).
# fsolve finds one root per starting point; numpy finds all four at once.

import matplotlib.pyplot as plt
import numpy as np
from numpy.polynomial import Polynomial
from scipy.optimize import fsolve


def complex_formatter(x):
    """Print complex numbers with zero imaginary part as plain reals."""
    if np.iscomplexobj(x) and np.imag(x) == 0:
        return f"{np.real(x):.4f}"
    return f"{np.round(x, 4)}"


np.set_printoptions(
    formatter={
        "float_kind": lambda x: f"{x:.4f}",
        "complex_kind": complex_formatter,
    }
)


def f(x):
    return x**4 + x - 1


x = np.linspace(-1.5, 1.5, 100)
plt.plot(x, f(x))
plt.title("$f(x)=x^4+x-1$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.axhline(0, color="black")
plt.grid(True)
plt.tight_layout()
plt.show()

# fsolve is a local solver: each call finds one root nearest the starting point.
# Two separate calls with different starting points reveal two real roots.
print(f"Roots via fsolve   : {fsolve(f, -1.5)}, {fsolve(f, 0.5)}")

# numpy finds all four roots of the degree-4 polynomial at once:
#   x^4 + x - 1  =>  coefficients [-1, 1, 0, 0, 1]
roots = Polynomial([-1, 1, 0, 0, 1]).roots()
print(f"Roots via numpy    : {roots}")
print(f"Valid roots (f=0)  : {roots[np.isclose(f(roots), 0)]}")

**Extraneous roots from algebraic manipulation.**
For $g(x) = -x^2 + x^{3/2} + 5x - 6$, the fractional exponent means
this is not a polynomial in $x$.
To apply polynomial root-finding, substitute $u = \sqrt{x}$ (so $x = u^2$)
and rearrange to isolate the radical:

$$
x^{3/2} = x^2 - 5x + 6
\quad\Rightarrow\quad
x^3 = (x^2-5x+6)^2 = x^4 - 11x^3 + 37x^2 - 60x + 36
$$

giving the degree-4 polynomial $x^4 - 11x^3 + 37x^2 - 60x + 36 = 0$.
Squaring introduces two extraneous roots that satisfy the polynomial
but not the original $g(x)$, so verification is essential.

In [ ]:
# Cell 02 - g(x) = -x^2 + x^(3/2) + 5x - 6
# Rationalized to a degree-4 polynomial by squaring.
# Two of the four polynomial roots are extraneous (introduced by squaring).

from matplotlib.ticker import MultipleLocator


def g(x):
    return -(x**2) + x ** (3 / 2) + 5 * x - 6


x = np.linspace(0, 10, 100)
plt.plot(x, g(x))
plt.title(r"$g(x)=-x^2+x^{3/2}+5x-6$")
plt.xlabel("x")
plt.ylabel("g(x)")
plt.axhline(0, color="black")
plt.gca().xaxis.set_major_locator(MultipleLocator(1))
plt.grid(True)
plt.tight_layout()
plt.show()

# fsolve finds the two real roots of g(x) directly
print(f"Roots via fsolve   : {fsolve(g, 1)}, {fsolve(g, 6)}")

# The rationalized polynomial has four roots, but only two satisfy g(x)=0
# x^4 - 11x^3 + 37x^2 - 60x + 36  =>  coefficients [36, -60, 37, -11, 1]
roots = Polynomial([36, -60, 37, -11, 1]).roots()
print(f"Roots via numpy    : {roots}")
print(f"Valid roots (g=0)  : {roots[np.isclose(g(roots), 0)]}")

**Irrational exponents via substitution.**
For $h(x) = x^{3.4} + x - 1 = x^{17/5} + x - 1$, neither squaring nor
simple rationalization helps directly.
Instead, substitute $u = x^{1/5}$ so $x = u^5$ and $x^{17/5} = u^{17}$:

$$
u^{17} + u^5 - 1 = 0
$$

This degree-17 polynomial has 17 roots in $u$, most complex.
Converting back via $x = u^5$ and verifying against the original $h(x)$
recovers the single real root.
Note that $h(x)$ is only defined for $x \geq 0$ in the reals
(fractional powers of negative numbers are complex), so only
non-negative real roots are physically meaningful.

In [ ]:
# Cell 03 - h(x) = x^(17/5) + x - 1
# Substitute u = x^(1/5): polynomial is u^17 + u^5 - 1 = 0
# One real root; convert back with x = u^5 and verify.


def h(x):
    return x**3.4 + x - 1


x = np.linspace(0, 2, 100)
plt.plot(x, h(x))
plt.title("$h(x)=x^{3.4}+x-1$")
plt.xlabel("x")
plt.ylabel("h(x)")
plt.axhline(0, color="black")
plt.grid(True)
plt.tight_layout()
plt.show()

# fsolve finds the single real root directly
print(f"Root via fsolve    : {fsolve(h, 1)}")

# u^17 + u^5 - 1 = 0  =>  coefficients [-1, 0, 0, 0, 0, 1, 0, ..., 0, 1]
coeffs = [0] * 18
coeffs[0] = -1  # constant term
coeffs[5] = 1  # u^5 term
coeffs[17] = 1  # u^17 term
u_roots = Polynomial(coeffs).roots()

# Convert u roots back to x = u^5, keep only real non-negative results
real_u = u_roots[np.abs(np.imag(u_roots)) < 1e-8]
x_roots = np.real(real_u) ** 5
x_roots = x_roots[x_roots >= 0]

print(f"u roots via numpy  : {u_roots}")
print(f"Valid root (h=0)   : {x_roots[np.isclose(h(x_roots), 0)]}")